# Fairness and Ethics Basics - Responsible Predictive Analytics

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/17_fairness_slicing_model_cards.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Identify fairness risks and ethical failure modes in predictive systems
2. Compute basic group fairness diagnostics (when sensitive attributes exist)
3. Use slicing to detect performance disparities across segments
4. Write a model card-style limitations and responsible-use section
5. Apply responsible AI framing to the course project deliverable

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: Does the Model Work for Everyone?

Before the screening tool goes live, the **State Health Department** ethics review board raises a final concern: *"Does this model perform equally well for all patient subgroups?"* You investigate and discover that recall for one demographic group is 8 percentage points lower than for another.

A model that's 97% accurate overall but misses cancers disproportionately in one group is not just a technical problem — it's an ethical and legal liability. The board requires a Model Card: a standardized document that transparently reports performance across subgroups, intended use, and known limitations.

> **Today's focus:** Slicing model performance by subgroup, quantifying fairness gaps, and creating a Model Card that documents the screening tool's capabilities and limitations.

> **A question that often comes up here:** *"If my model's overall AUC is 0.95, isn't it already good?"* Not necessarily. An aggregate 0.95 is the average across all groups, and averages can hide dramatic disparities. A model that achieves 0.98 on majority-group patients and 0.72 on minority-group patients still averages to about 0.95 on a mostly-majority dataset — but the 0.72 is a real-world harm. Slice-based evaluation (this notebook's core tool) is what catches that. You are not introducing politics into modeling by slicing; you are refusing to let the aggregate hide known failures.

---


## 1. Setup

Before the screening tool rolls out to partner hospitals, the Health Department's ethics review board wants proof that the model performs equitably across patient subgroups. This cell loads the classification metrics needed for sliced evaluation (`accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix`) alongside standard data-wrangling and plotting libraries. We lock `RANDOM_SEED = 474` so that the synthetic group assignments and train/test splits are identical every time the notebook is run — critical for a fairness audit, where even small changes in the random split can flip the direction of a measured disparity.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print("✓ Setup complete!")

**Reading the output:**

The `Setup complete!` line confirms all imports loaded without error and the random seed is locked at **474**. The full set of per-group metrics (`accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix`) is available, which means we can slice performance by any categorical attribute and compute fairness diagnostics.

**Why this matters:** Reproducibility is especially important in fairness analysis because the ethics review board will want to re-run this audit independently. If different random seeds produce different disparity conclusions, the audit is unreliable. Locking the seed and documenting the exact pipeline configuration ensures the Health Department can defend its fairness assessment to regulators, hospital boards, and community advocates.

---

## 2. Fairness Vocabulary

Before measuring disparities, the Health Department's audit team needs a shared vocabulary. These concepts recur throughout the screening tool's ethics review and Model Card documentation:

### Key Concepts:

**Disparity**: Systematic difference in outcomes across groups — e.g., the model misses cancer at a higher rate for one demographic group than another.

**Harm**: Negative impact on individuals or groups
- **Allocation harm**: Withholding resources — a patient flagged as "low risk" may not receive the follow-up biopsy they need
- **Quality-of-service harm**: Worse predictions for some groups — the screening tool may achieve 95% recall for Group A but only 87% for Group B
- **Representational harm**: Stereotyping or denigration — a model trained primarily on one population may pathologise normal variation in another

**Proxy variables**: Features correlated with sensitive attributes — in the screening context, hospital location or insurance type might correlate with demographics even though they are not explicitly demographic features

**Feedback loops**: Model decisions influence future data, potentially amplifying bias — if the tool under-screens a group, fewer cancers are caught in that group, which makes the training data even more imbalanced for the next retraining cycle

### Common Fairness Metrics:

1. **Demographic Parity**: Equal selection rate across groups  
   P(ŷ=1 | A=a) = P(ŷ=1 | A=b)

2. **Equal Opportunity**: Equal true positive rate across groups  
   P(ŷ=1 | y=1, A=a) = P(ŷ=1 | y=1, A=b)

3. **Equalized Odds**: Equal TPR and FPR across groups

⚠️ **Important**: These metrics often conflict with each other! The Health Department must decide which fairness definition matters most for cancer screening — equal detection rates (equal opportunity) or equal referral rates (demographic parity).

---

## 3. Generate Synthetic Data with Group Attribute

For this fairness exercise, we use `make_classification` to create 2,000 samples with 15 features (10 informative, 5 redundant) and a 60/40 positive-negative class split. A synthetic `group` column is added *after* feature generation — it is randomly assigned (60% Group_A, 40% Group_B) and deliberately **not** used as a training feature. This mirrors the screening scenario where demographic information exists in the patient records but is excluded from the model's input features.

Even though the group attribute is excluded from training, the model may still produce different performance across groups if the features correlate with group membership — exactly the kind of indirect discrimination the Health Department's audit is designed to detect.

> 💡 **Gemini Prompt:** "Using make_classification, generate 2000 samples, 15 features (10 informative, 5 redundant), 60/40 imbalance. Add synthetic 'group' column (Group_A 60%, Group_B 40%) NOT used as feature. Show shape, group distribution, cross-tab."
>
> **After running, verify:**
> - 2000 rows, 17 columns (15 features + target + group)
> - Group_A \~60%, Group_B \~40%
> - Cross-tab shows target proportions per group
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Generate classification data
X, y = make_classification(
    n_samples=2000, n_features=15, n_informative=10,
    n_redundant=5, weights=[0.6, 0.4],
    random_state=RANDOM_SEED
)

# Add synthetic "group" attribute (not used in training)
# Group A: 60%, Group B: 40%
group = np.random.choice(['Group_A', 'Group_B'], size=len(X), p=[0.6, 0.4])

# Create DataFrame
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['group'] = group

print(f"Dataset shape: {df.shape}")
print(f"\nGroup distribution:")
print(df['group'].value_counts())
print(f"\nTarget distribution by group:")
print(pd.crosstab(df['group'], df['target'], normalize='index'))

**Reading the output:**

The dataset has **2,000 rows** and **17 columns** (15 features + target + group). The group distribution should be approximately **60% Group_A** and **40% Group_B**. The cross-tabulation of target by group shows whether the positive-class base rate differs between groups — this is critical context for the audit.

If the base rate differs (e.g., 42% positive in Group_A vs. 38% in Group_B), even a perfectly calibrated model will produce different selection rates across groups. This is the root of many fairness dilemmas in healthcare: if cancer prevalence genuinely differs across populations, requiring equal screening rates across groups may mean under-screening the higher-prevalence group or over-screening the lower-prevalence group.

**Key takeaway:** Understanding base rates by group *before* modelling is essential for the Health Department's audit. A difference in base rates does not automatically mean the model is unfair, but it does mean that equal selection rates and equal accuracy cannot both be achieved simultaneously — the audit board must choose which fairness definition to prioritise.

---

## 4. Train Model (Without Using Group Attribute)

The screening tool is trained on the 15 numeric features only — the `group` column is deliberately excluded, just as the Health Department would exclude patient demographics from the model's inputs. The train/test split is stratified by group to ensure both sets contain proportional representation. After training, we compute overall accuracy, precision, recall, and F1 on the test set as an aggregate baseline before drilling into group-level performance.

The question is not whether the model uses the group attribute directly (it does not), but whether it achieves different outcomes for different groups through correlated features — the definition of indirect discrimination.

> 💡 **Gemini Prompt:** "Split 70/30 stratified by group. Train RandomForest (100 trees, max_depth=10) on features only (exclude target and group). Print overall accuracy, precision, recall, F1."
>
> **After running, verify:**
> - Train \~1400, test \~600
> - Model trained WITHOUT group column
> - Overall metrics printed
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Split data (stratified by group to ensure representation)
train_df, test_df = train_test_split(df, test_size=0.3, stratify=df['group'], random_state=RANDOM_SEED)

# Separate features, target, and group
X_train = train_df[feature_names]
y_train = train_df['target']
group_train = train_df['group']

X_test = test_df[feature_names]
y_test = test_df['target']
group_test = test_df['group']

# Train model WITHOUT group attribute
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n=== OVERALL MODEL PERFORMANCE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")

**Reading the output:**

The overall **accuracy**, **precision**, **recall**, and **F1** are computed on the full test set without any group breakdown. These aggregate numbers tell the Health Department how the screening tool performs *on average* across all patients. A strong overall F1 does not guarantee equitable performance — the model might achieve 95% recall for one group while missing 13% more cancers in another.

**Why this matters:** Reporting only aggregate metrics is the most common pitfall in responsible AI. The Health Department's ethics board will not accept "95% recall overall" if it masks a significant recall gap between demographic groups. The sliced evaluation in the next section is what the audit is really about.

---

## 5. Slice-Based Performance Evaluation

### 5.1 Performance by Group

Aggregate metrics can mask life-or-death disparities. If the screening tool catches 95% of cancers in Group_A but only 87% in Group_B, patients in Group_B face systematically worse outcomes — and the Health Department faces legal and ethical liability. Slicing the test set by `group` and computing accuracy, precision, recall, and F1 separately for each segment reveals whether the model serves all patients equitably.

> 💡 **Gemini Prompt:** "For each group in test set, compute accuracy, precision, recall, F1. Display per-group DataFrame and print performance gaps (Group_A minus Group_B)."
>
> **After running, verify:**
> - DataFrame shows per-group metrics
> - Disparity values with +/- sign
> - Gap > 0.05 signals fairness concern
> - Bar plot clearly visualizes the comparison across models/groups
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compute metrics by group
slice_results = []

for grp in ['Group_A', 'Group_B']:
    mask = group_test == grp
    y_true_grp = y_test[mask]
    y_pred_grp = y_pred[mask]
    
    slice_results.append({
        'Group': grp,
        'Sample_Size': len(y_true_grp),
        'Accuracy': accuracy_score(y_true_grp, y_pred_grp),
        'Precision': precision_score(y_true_grp, y_pred_grp, zero_division=0),
        'Recall': recall_score(y_true_grp, y_pred_grp, zero_division=0),
        'F1': f1_score(y_true_grp, y_pred_grp, zero_division=0)
    })

slice_df = pd.DataFrame(slice_results)

print("\n=== PERFORMANCE BY GROUP ===")
print(slice_df.to_string(index=False))

# Compute disparities
print("\n=== PERFORMANCE DISPARITIES ===")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
    gap = slice_df.loc[0, metric] - slice_df.loc[1, metric]
    print(f"{metric} Gap (A - B): {gap:+.4f}")
# --- Bar plot: Metrics by group ---
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1']
fig, axes = plt.subplots(1, len(metrics_cols), figsize=(16, 4))
for ax, metric in zip(axes, metrics_cols):
    slice_df.plot.bar(x='Group', y=metric, ax=ax, legend=False, color=['steelblue', 'coral'])
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9)
fig.suptitle('Performance Metrics by Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The **Performance by Group** table shows accuracy, precision, recall, and F1 for Group_A and Group_B side by side. The **Performance Disparities** section computes the signed gap (A minus B) for each metric. A positive gap means Group_A scores higher; a negative gap means Group_B does.

Pay special attention to the **Recall gap**: in cancer screening, recall is the metric that directly measures whether the tool catches malignancies. A recall gap of 0.05 means that for every 100 true cancers in each group, Group_B's model misses 5 more patients than Group_A's. Across 10,000 annual screenings, that gap translates to dozens of delayed diagnoses in the underperforming group.

**Key takeaway:** Even small metric gaps compound at scale. The Health Department must assess both the magnitude of the gap and the sample sizes to determine whether the disparity is statistically meaningful and practically harmful. A 0.03 recall gap on 50 test samples is noise; the same gap on 5,000 samples is a systemic problem.

---

> 💡 **Gemini Prompt:** "Create grouped bar chart: Group_A vs Group_B across accuracy, precision, recall, F1. Side-by-side bars, y-axis 0-1, legend, grid."
>
> **After running, verify:**
> - Two colored bars per metric
> - Y-axis 0-1 with gridlines
> - Visual gaps reveal unfairness
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Visualize performance gaps
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, slice_df.loc[0, metrics], width, label='Group A', alpha=0.8)
ax.bar(x + width/2, slice_df.loc[1, metrics], width, label='Group B', alpha=0.8)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance by Group')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Reading the output:**

The grouped bar chart plots accuracy, precision, recall, and F1 side by side for Group_A and Group_B. Bars of similar height indicate comparable performance; a visible height difference on any metric flags a potential disparity. Because the y-axis runs from 0 to 1, even a few percentage points of difference is easy to spot visually.

**Why this matters:** This chart is the centrepiece of the Health Department's fairness audit presentation to the ethics board. A bar chart showing one group's recall noticeably shorter than the other communicates the issue far more effectively than a table of numbers. Community advocates and patient representatives who review the audit will look at this chart first — it must be clear, honest, and annotated with the gap values.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Create slice performance table and highlight one disparity (if any).

**Instructions:**
1. Review the slice performance results above
2. Identify the largest performance gap between groups
3. Assess whether this gap is meaningful/concerning
4. Propose one hypothesis for why the gap exists

---

### YOUR DISPARITY ANALYSIS HERE:

**Largest Disparity:**  
[Which metric and how large?]

**Is it Concerning?:**  
[Your assessment with reasoning]

**Hypothesis:**  
[Why might this disparity exist?]

---

## 6. Fairness Metrics

### 6.1 Selection Rate (Demographic Parity)

Beyond accuracy and recall, the Health Department's audit examines three formal fairness metrics. The **selection rate** measures demographic parity: are both groups flagged for biopsy at the same rate? The **TPR gap** measures equal opportunity: among patients who truly have cancer, does the model catch them at the same rate regardless of group? The **FPR gap** completes equalized odds: among healthy patients, are both groups equally likely to receive a false alarm?

> 💡 **Gemini Prompt:** "For each group, compute selection rate, TPR, FPR from confusion matrix. Display fairness metrics DataFrame and print gaps with interpretation."
>
> **After running, verify:**
> - DataFrame shows selection rate, TPR, FPR per group
> - Gaps printed with signs
> - Interpretation explains each gap's fairness meaning
> - Bar plot clearly visualizes the comparison across models/groups
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compute selection rates
fairness_metrics = []

for grp in ['Group_A', 'Group_B']:
    mask = group_test == grp
    y_pred_grp = y_pred[mask]
    y_true_grp = y_test[mask]
    
    # Selection rate
    selection_rate = y_pred_grp.mean()
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true_grp, y_pred_grp).ravel()
    
    # Rates
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # True Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    
    fairness_metrics.append({
        'Group': grp,
        'Selection_Rate': selection_rate,
        'TPR': tpr,
        'FPR': fpr
    })

fairness_df = pd.DataFrame(fairness_metrics)

print("\n=== FAIRNESS METRICS ===")
print(fairness_df.to_string(index=False))

# Compute fairness gaps
print("\n=== FAIRNESS GAPS ===")
sr_gap = fairness_df.loc[0, 'Selection_Rate'] - fairness_df.loc[1, 'Selection_Rate']
tpr_gap = fairness_df.loc[0, 'TPR'] - fairness_df.loc[1, 'TPR']
fpr_gap = fairness_df.loc[0, 'FPR'] - fairness_df.loc[1, 'FPR']

print(f"Selection Rate Gap: {sr_gap:+.4f}")
print(f"TPR Gap (Equal Opportunity): {tpr_gap:+.4f}")
print(f"FPR Gap: {fpr_gap:+.4f}")

print("\n⚠️ Interpretation:")
print("  - Selection Rate: Are both groups selected at similar rates?")
print("  - TPR Gap: Do qualified individuals have equal chance across groups?")
print("  - FPR Gap: Are false positives distributed equally?")
# --- Bar plot: Fairness metrics by group ---
fairness_df = pd.DataFrame(fairness_metrics)
metrics_cols = ['Selection_Rate', 'TPR', 'FPR']
fig, axes = plt.subplots(1, len(metrics_cols), figsize=(14, 4))
for ax, metric in zip(axes, metrics_cols):
    fairness_df.plot.bar(x='Group', y=metric, ax=ax, legend=False, color=['steelblue', 'coral'])
    ax.set_title(metric.replace('_', ' '), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9)
fig.suptitle('Fairness Metrics by Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

Three fairness diagnostics are printed per group. The **Selection Rate** measures demographic parity: if both groups are flagged for biopsy at the same rate, the gap is near zero. The **TPR gap** captures equal opportunity: whether truly malignant cases are caught at equal rates regardless of group — this is the metric the oncologist cares about most. The **FPR gap** measures whether healthy patients in both groups face equal false-alarm rates — this is what drives biopsy costs and patient anxiety.

**Key takeaway:** These three metrics often conflict. For cancer screening, the Health Department should prioritise **equal opportunity** (same recall across groups) over **demographic parity** (same flagging rate). Missing cancer in any subgroup is the greater harm, and the screening programme must demonstrate equitable detection across all populations it serves. The Model Card in the next section will document which definition was chosen and why.

---

## 7. Model Card Template

### 7.1 Model Card Structure

A Model Card is the screening tool's "nutrition label" — a short, structured document that accompanies the trained model and discloses its intended use, training data, performance metrics (including per-group breakdowns), limitations, and ethical considerations. The Health Department's ethics review board requires one before approving deployment to partner hospitals.

The template below follows the format proposed by Mitchell et al. (2019). Filling it out forces the team to confront potential harms before the tool touches a single patient.

# Model Card: [Your Model Name]

## Model Details
- **Model Type**: Random Forest Classifier
- **Version**: 1.0
- **Date**: June 9, 2027
- **Developed By**: [Your name/team]

## Intended Use
**Primary Use Cases:**
- [Describe primary intended application]
- [Context where model should be used]

**Out-of-Scope Uses:**
- [Contexts where model should NOT be used]
- [Scenarios where model may fail or cause harm]

## Training Data
- **Source**: [Dataset name and source]
- **Size**: [Number of samples]
- **Time Period**: [When data was collected]
- **Geography**: [Where data was collected]
- **Preprocessing**: [Key preprocessing steps]

## Evaluation Data
- **Source**: [Test set details]
- **Size**: [Number of samples]
- **Split Method**: [How test set was created]

## Metrics
**Overall Performance:**
- Accuracy: X.XX
- Precision: X.XX
- Recall: X.XX
- F1: X.XX

**Performance by Group:**
- Group A: [Metrics]
- Group B: [Metrics]
- Performance Gap: [Description]

## Limitations
1. **Data Limitations**:
   - [Describe data quality issues, biases, or gaps]
   - [Temporal or geographic limitations]

2. **Model Limitations**:
   - [Known failure modes or weak segments]
   - [Assumptions that may not hold]
   - [Performance disparities across groups]

3. **Deployment Limitations**:
   - [Contexts where model should not be used]
   - [Required human oversight]
   - [Monitoring requirements]

## Ethical Considerations
**Potential Harms:**
- [Allocation harms - who might be denied opportunities?]
- [Quality-of-service harms - who might receive worse predictions?]
- [Feedback loop risks - how might the model affect future data?]

**Mitigation Strategies:**
- [What steps are taken to reduce harm?]
- [How are performance disparities addressed?]
- [What oversight mechanisms exist?]

## Recommendations
1. **Usage Recommendations**:
   - [How should model outputs be used?]
   - [What human review is required?]

2. **Monitoring**:
   - [What metrics should be tracked over time?]
   - [How often should model be re-evaluated?]

3. **Update Schedule**:
   - [When should model be retrained?]
   - [What triggers retraining?]

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Draft a model card limitations section (6-8 lines).

**Instructions:**
1. Review the model card template above
2. Write a limitations section for your project model
3. Include data limitations, model limitations, and ethical considerations
4. Be specific and evidence-based

---

### YOUR MODEL CARD LIMITATIONS SECTION:

## Limitations

1. **Data Limitations**:  
[Your text here - be specific]

2. **Model Limitations**:  
[Your text here - reference specific performance issues]

3. **Ethical Considerations**:  
[Your text here - discuss potential harms]

---

## 8. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Aggregate metrics hide group-level failures.** An overall AUC of 0.95 can mask a 20-point disparity between groups. Slice-based evaluation — computing the same metric per demographic slice — is the minimum-viable fairness audit.
2. **Removing the sensitive attribute does not remove its influence.** Proxies (income, zip code, education) carry the same information. Fairness audits must be done on predictions and outcomes, not on input column lists.
3. **Multiple fairness metrics exist and they are incompatible in general.** Demographic parity (equal selection rates), equal opportunity (equal TPR across groups), and calibration (predicted probability = true rate per group) cannot all hold unless base rates are equal. Pick the criterion that matches the deployment's stakes; document the tradeoff.
4. **Model cards are the artifact that makes all of this auditable.** A model card compresses the modeling work (champion, CV, test set), the interpretation work (NB15), the calibration work (NB16), and the fairness work (this notebook) into one signed-off document — including the limitations, the intended-use boundaries, and the monitoring plan.

### Critical Rules:

> **"Slice metrics by group — always, every time, no exceptions."**

> **"Removing the sensitive attribute does not remove the proxy problem."**

> **"Pick a fairness criterion that matches the stakes; document the tradeoff explicitly."**

### Next Steps:

- **NB18 (Reproducibility + Monitoring + Kaggle submission)** turns the model you validated into something deployable — save/load pipelines with joblib, a monitoring plan with drift signals, and the last-mile mechanics for producing a `submission.csv` for the Kaggle competition deadline.
- **NB19 (Executive Narrative)** turns the model card into a slide narrative — the five-act structure that a non-technical stakeholder can follow.
- **Your final project's model card** should use NB17's template. If you cannot fill in the limitations and intended-use sections, you are not ready to deploy — which is the point of the exercise.

> **A question that often comes up here:** *"If the fairness audit reveals a disparity I cannot fix without retraining, do I have to retrain?"* Not necessarily. Sometimes the right response is *"we document the disparity, restrict the model's use to contexts where the disparity does not translate into harm, and monitor it in production."* Model cards are the place for that documentation. Hiding disparities is the professional failure mode; documenting them honestly — even when the fix is out of scope — is not.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Barocas, S., Hardt, M., & Narayanan, A. (2019). *Fairness and Machine Learning*. [fairmlbook.org](http://fairmlbook.org/)
- Hardt, M., Price, E., & Srebro, N. (2016). "Equality of Opportunity in Supervised Learning." *NeurIPS*.
- Mitchell, M., et al. (2019). "Model Cards for Model Reporting." *FAT**.
- Chouldechova, A. (2017). "Fair prediction with disparate impact." *Big Data*.
- Selbst, A.D., et al. (2019). "Fairness and Abstraction in Sociotechnical Systems." *FAT**.

---




<center>

Thank you!

</center>